# InstructABSA — Train + Evaluate trên Google Colab T4

Notebook này huấn luyện InstructABSA (ATSC subtask, instruction Set-2) và đánh giá trên 3 dataset: SemEval-14 Restaurant, SemEval-14 Laptop, UIT-VSFC.

**Quy trình:**
1. Mount Google Drive (lưu checkpoint persistent).
2. Upload `instruct_absa_bundle.zip` (tạo bằng `python scripts/make_colab_bundle.py` ở local).
3. Cài deps tối thiểu.
4. Train 3 model (rest → lap → vsfc), checkpoint lưu vào Drive.
5. Evaluate 3 model qua `evaluate.py --given-aspect`.
6. Zip + tải `results/` về máy.

**Backbone:**
- EN: `allenai/tk-instruct-base-def-pos` (220M)
- VI: `google/mt5-base` (580M)

**Hardware:** Colab T4 16GB, fp16, batch 4 (EN) / 2 (VI), grad accum 4/8.

## 1. Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

## 2. Mount Google Drive

Checkpoint sẽ lưu tại `/content/drive/MyDrive/instruct_absa_ckpts/` để không mất khi runtime timeout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CKPT_ROOT = '/content/drive/MyDrive/instruct_absa_ckpts'
os.makedirs(DRIVE_CKPT_ROOT, exist_ok=True)
print('Drive checkpoint root:', DRIVE_CKPT_ROOT)

## 3. Upload bundle và giải nén

Trên máy local, chạy: `python scripts/make_colab_bundle.py` → tạo `instruct_absa_bundle.zip`. Upload zip này khi cell dưới chạy.

In [ ]:
from google.colab import files
uploaded = files.upload()  # chọn instruct_absa_bundle.zip

import zipfile, os
WORK_DIR = '/content/absa-sota-survey'
os.makedirs(WORK_DIR, exist_ok=True)
for name in uploaded:
    with zipfile.ZipFile(name, 'r') as zf:
        zf.extractall(WORK_DIR)
    print('Extracted', name, '->', WORK_DIR)

%cd /content/absa-sota-survey
!ls

## 4. Cài đặt thư viện tối thiểu

Colab đã có torch + numpy. Chỉ cần cập nhật transformers + sentencepiece (cho mT5).

In [ ]:
!pip install -q -U transformers accelerate sentencepiece pyyaml

## 5. Symlink checkpoint dir → Drive

Để train script ghi vào Drive (persistent) thay vì `/content/` (mất khi timeout).

In [ ]:
import os, pathlib

for ds_slug in ('semeval14_rest', 'semeval14_lap', 'vsfc'):
    drive_dir = f'{DRIVE_CKPT_ROOT}/{ds_slug}'
    os.makedirs(drive_dir, exist_ok=True)
    local_link = f'/content/absa-sota-survey/checkpoints/{ds_slug}'
    pathlib.Path(local_link).parent.mkdir(parents=True, exist_ok=True)
    if os.path.islink(local_link) or os.path.exists(local_link):
        if not os.path.islink(local_link):
            print(f'[WARN] {local_link} đã tồn tại nhưng không phải symlink; bỏ qua.')
            continue
        os.remove(local_link)
    os.symlink(drive_dir, local_link)
    print(f'symlink {local_link} -> {drive_dir}')

## 6. Train — SemEval-14 Restaurant (EN)

Tk-Instruct-base-def-pos 220M, batch 4, grad accum 4, 4 epochs, fp16. Ước lượng ~15-20 phút.

In [ ]:
!python -m models.instruct_absa.train --config configs/instruct_absa_en_restaurant.yaml

## 7. Train — SemEval-14 Laptop (EN)

In [ ]:
!python -m models.instruct_absa.train --config configs/instruct_absa_en_laptop.yaml

## 8. Train — UIT-VSFC (VI)

mT5-base 580M, batch 2, grad accum 8, 4 epochs. Ước lượng ~30-40 phút (data lớn nhất + model nặng nhất).

In [ ]:
!python -m models.instruct_absa.train --config configs/instruct_absa_vi.yaml

## 9. Evaluate — Restaurant (EN)

In [ ]:
!python evaluate.py \
    --predictor predictors.instruct_absa:InstructABSAPredictor \
    --predictor-kwargs '{"checkpoint":"checkpoints/semeval14_rest/instruct_absa_best","language":"en"}' \
    --test-set data/processed/lcf_bert/semeval14_rest_test.jsonl \
    --given-aspect --output-dir results

## 10. Evaluate — Laptop (EN)

In [ ]:
!python evaluate.py \
    --predictor predictors.instruct_absa:InstructABSAPredictor \
    --predictor-kwargs '{"checkpoint":"checkpoints/semeval14_lap/instruct_absa_best","language":"en"}' \
    --test-set data/processed/lcf_bert/semeval14_lap_test.jsonl \
    --given-aspect --output-dir results

## 11. Evaluate — UIT-VSFC (VI)

In [ ]:
!python evaluate.py \
    --predictor predictors.instruct_absa:InstructABSAPredictor \
    --predictor-kwargs '{"checkpoint":"checkpoints/vsfc/instruct_absa_best","language":"vi"}' \
    --test-set data/processed/lcf_bert/vsfc_test.jsonl \
    --given-aspect --output-dir results

## 12. Đóng gói results và tải về máy

In [ ]:
import shutil
shutil.make_archive('/content/instruct_absa_results', 'zip', '/content/absa-sota-survey/results')
print('Created /content/instruct_absa_results.zip')

from google.colab import files
files.download('/content/instruct_absa_results.zip')

## 13. (Tuỳ chọn) In nhanh metrics ngay tại Colab

In [ ]:
import json, glob
for p in sorted(glob.glob('/content/absa-sota-survey/results/metrics/instruct_absa_*.json')):
    m = json.load(open(p))
    print(f"{m['method']:14s} {m['dataset']:30s} "
          f" sent_acc={m['sentiment_accuracy']:.4f}"
          f"  macro_f1={m['sentiment_macro_f1']:.4f}"
          f"  latency={m['avg_latency_ms']}ms")